In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE_DIR = Path().resolve().parent
RAW = BASE_DIR / "data" / "raw"

app_train = pd.read_csv(RAW / "application_train.csv")
bureau = pd.read_csv(RAW / "bureau.csv")
installments = pd.read_csv(RAW / "installments_payments.csv")

print(f"Application Train: {app_train.shape}")
print(f"Bureau: {bureau.shape}")
print(f"Installments: {installments.shape}")

Application Train: (307511, 122)
Bureau: (1716428, 17)
Installments: (13605401, 8)


In [2]:
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [3]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT        = ('SK_ID_BUREAU', 'count'),
    BUREAU_CREDIT_DAY_OVERDUE_MEAN = ('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_AMT_CREDIT_SUM_MEAN     = ('AMT_CREDIT_SUM', 'mean'),
    BUREAU_AMT_CREDIT_SUM_MAX      = ('AMT_CREDIT_SUM', 'max'),
    BUREAU_ACTIVE_LOANS      = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum())
).reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 6)


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_ACTIVE_LOANS
0,100001,7,0.0,207623.571429,378000.0,3
1,100002,8,0.0,108131.945625,450000.0,2
2,100003,4,0.0,254350.125000,810000.0,1
3,100004,2,0.0,94518.900000,94537.8,0
4,100005,3,0.0,219042.000000,568800.0,2


In [4]:

installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
installments['DAYS_LATE'] = installments['DAYS_LATE'].clip(lower=0)  


installments_agg = installments.groupby('SK_ID_CURR').agg(
    INSTALLMENTS_COUNT          = ('SK_ID_PREV', 'count'),
    INSTALLMENTS_AMT_PAYMENT_MEAN = ('AMT_PAYMENT', 'mean'),
    INSTALLMENTS_AMT_PAYMENT_MAX  = ('AMT_PAYMENT', 'max'),
    INSTALLMENTS_DAYS_LATE_MEAN   = ('DAYS_LATE', 'mean'),
    INSTALLMENTS_DAYS_LATE_MAX    = ('DAYS_LATE', 'max')
).reset_index()

print(installments_agg.shape)
installments_agg.head()

(339587, 6)


,SK_ID_CURR,INSTALLMENTS_COUNT,INSTALLMENTS_AMT_PAYMENT_MEAN,INSTALLMENTS_AMT_PAYMENT_MAX,INSTALLMENTS_DAYS_LATE_MEAN,INSTALLMENTS_DAYS_LATE_MAX
0,100001,7,5885.132143,17397.900,1.571429,11.0
1,100002,19,11559.247105,53093.745,0.000000,0.0
2,100003,25,64754.586000,560835.360,0.000000,0.0
3,100004,3,7096.155000,10573.965,0.000000,0.0
4,100005,9,6240.205000,17656.245,0.111111,1.0


In [5]:
# merging app_train with bureau_agg and installments_agg
data = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
data = data.merge(installments_agg, on='SK_ID_CURR', how='left')
print(data.shape)

(307511, 132)


In [7]:


cols_to_drop = [col for col in data.columns if data[col].isnull().mean() > 0.5]
print(f"Dropping {len(cols_to_drop)} columns with >50% missing values")
data = data.drop(columns=cols_to_drop)


for col in data.columns:
    if data[col].dtype in ['float64', 'int64']:
        data[col] = data[col].fillna(data[col].median())
    else:
        data[col] = data[col].fillna(data[col].mode()[0])


print(f"Missing values remaining: {data.isnull().sum().sum()}")
print(f"Final shape: {data.shape}")

Dropping 0 columns with >50% missing values
Missing values remaining: 0
Final shape: (307511, 91)


In [8]:

data['CREDIT_INCOME_RATIO'] = data['AMT_CREDIT'] / data['AMT_INCOME_TOTAL']
  
data['ANNUITY_INCOME_RATIO'] = data['AMT_ANNUITY'] / data['AMT_INCOME_TOTAL']

data['CREDIT_ANNUITY_RATIO'] = data['AMT_CREDIT'] / data['AMT_ANNUITY']

data['INCOME_PER_PERSON'] = data['AMT_INCOME_TOTAL'] / data['CNT_FAM_MEMBERS']

In [9]:

categorical_cols = data.select_dtypes(include=['object']).columns


data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding: {data.shape}")
print(f"New columns added: {data.shape[1] - 91}")

Shape after encoding: (307511, 195)
New columns added: 104


In [11]:
import os

processed_path = BASE_DIR / "data" / "processed"
os.makedirs(processed_path, exist_ok=True)

data.to_csv(processed_path / "processed_train.csv", index=False)

test_load = pd.read_csv(processed_path / "processed_train.csv")
print(f"Saved successfully: {test_load.shape}")

Saved successfully: (307511, 195)
